In [49]:
import os
import warnings
import seaborn as sns
import numpy as np
import pandas as pd
import openpyxl
import matplotlib.pyplot as plt

In [44]:
import torch
from sklearn.model_selection import LeaveOneGroupOut, ParameterGrid, KFold, GroupKFold, StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, balanced_accuracy_score, recall_score, precision_score, confusion_matrix
from xgboost import XGBClassifier

In [3]:
import catboost as cb
from catboost import CatBoostClassifier

In [4]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVR, SVC
# XGBoost
from xgboost import XGBClassifier
# CatBoost
from catboost import CatBoostClassifier
# LightGBM
from lightgbm import LGBMClassifier

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [6]:
import tensorflow as tf
from scikeras.wrappers import KerasClassifier

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tcn import TCN

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

In [41]:
from sklearn.linear_model import RidgeClassifier
from sklearn.pipeline import Pipeline

import numba
from sktime.transformations.panel.rocket import Rocket
from sklearn.base import clone
# alternativa più veloce spesso: MiniRocket (se disponibile nella tua versione)
# from sktime.transformations.panel.rocket import MiniRocket

In [ ]:
def build_lstm_model(n_features, n_units=64, dropout=0.2, lr=1e-3):

    model = Sequential([
        LSTM(64, input_shape=(timesteps, n_features)),
        Dense(32, activation="relu"),
        Dense(1, activation="sigmoid")
    ])
    
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    #model.summary()
    return model

In [8]:
def build_lstm_model_2(n_features, n_units=32, dropout=0.2, lr=1e-3):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(None, n_features)),  # (timesteps, features)
        tf.keras.layers.LSTM(n_units, dropout=dropout),
        tf.keras.layers.Dense(1, activation="sigmoid")  # binaria
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [ ]:
def build_tcn_model(n_features, n_filters=32, kernel_size=3, lr=1e-3):
    model = Sequential([
    TCN(
            nb_filters=64,
            kernel_size=3,
            dilations=[1, 2, 4, 8],
            input_shape=(timesteps, n_features)
        ),
            Dense(32, activation="relu"),
            Dense(1, activation="sigmoid")
                    ])
    model.compile(
                    optimizer="adam",
                    loss="binary_crossentropy",
                    metrics=["accuracy"]
                )
    
    #model.summary()
    return model

In [9]:
def build_tcn_model_2(n_features, n_filters=32, kernel_size=3, lr=1e-3):
    # TCN "semplice": Conv1D + pooling (non è una TCN completa stile paper, ma didattica e funziona)
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(None, n_features)),
        tf.keras.layers.Conv1D(filters=n_filters, kernel_size=kernel_size, padding="causal", activation="relu"),
        tf.keras.layers.Conv1D(filters=n_filters, kernel_size=kernel_size, padding="causal", activation="relu"),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [42]:
def inner_LoPo_ParOpt(model, param_grid, X_train_outer, y_train_outer, groups_train_outer):
    
    # Inner LOPO for Hyperparameter Optimization
    inner_logo = LeaveOneGroupOut()
    best_model = None
    best_score = -np.inf
    
    for inner_train_idx, inner_val_idx in inner_logo.split(X_train_outer, y_train_outer, groups_train_outer):
        X_train_inner, X_val = X_train_outer[inner_train_idx], X_train_outer[inner_val_idx]
        y_train_inner, y_val = y_train_outer[inner_train_idx], y_train_outer[inner_val_idx]
        groups_train_inner = groups_train_outer.iloc[inner_train_idx]
        #print(groups_train_inner.unique())
        
        # Hyperparameter tuning
        for params in ParameterGrid(param_grid):
            params = {k: int(v) if isinstance(v, np.generic) else v for k, v in params.items()}
            model_clone = clone(model)       # crea copia pulita del modello
            model_clone.set_params(**params)
            model.fit(X_train_inner, y_train_inner)
            y_val_pred = model.predict(X_val)
            score = f1_score(y_val, y_val_pred, average="macro")

            if score > best_score:
                best_score = score
                best_model = model
                best_params = params  # Store the best hyperparameters
                
    return best_model, best_params, best_score
    

In [11]:
def inner_GridSearch_ParOpt(estimator, param_grid, X_train_outer, y_train_outer, groups_train_outer):
    
    inner_cv = LeaveOneGroupOut()
    grid = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring="f1_macro",
        cv=inner_cv,
        n_jobs=-1
    )
    
    grid.fit(X_train_outer, y_train_outer, groups=groups_train_outer)
    
    return grid.best_estimator_, grid.best_params_, grid.best_score_

In [12]:
def save_ConfMat(overall_conf_matrix, root):

    # Plot overall confusion matrix
    plt.figure(figsize=(6, 5))
    sns.heatmap(overall_conf_matrix, annot=True, cmap="coolwarm", xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Overall Confusion Matrix OHSS")
    conf_matrix_path = os.path.join(root, "results/overall_confusion_matrix_ohss.pdf")
    plt.savefig(conf_matrix_path, dpi=300, bbox_inches='tight')
    plt.close()

    return 0

#print(f"\n✅ Performance metrics saved as: {output_path}")
#print(f"✅ Overall Confusion Matrix saved as: {conf_matrix_path}")

In [13]:
# Suppress warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
seed = 69
torch.manual_seed(seed)
np.random.seed(seed)

# Define root directory
root = '.'

In [14]:
df = pd.read_csv('./new_dataset/maison-llf-features.CSV', sep=",")

In [15]:
count_vec = CountVectorizer()

In [16]:
ana = pd.read_csv('./new_dataset/maison-llf-demographics.CSV', sep=",")

In [17]:
ana_col = list(ana.columns)

In [18]:
for column in ana_col:

    if column != 'age' and column != 'participant':

        #print(column)
        
        count_matrix = count_vec.fit_transform(ana[column])
        count_df = pd.DataFrame(
            count_matrix.toarray(),
            columns=[f"CountVect_{w}" for w in count_vec.get_feature_names_out()]
        )
        ana = pd.concat([ana, count_df], axis=1)

In [19]:
tfidf_vec = TfidfVectorizer()

In [20]:
for column in ana_col:

    if column != 'age' and column != 'participant':

        #print(column)
        
        count_matrix = tfidf_vec.fit_transform(ana[column])
        count_df = pd.DataFrame(
            count_matrix.toarray(),
            columns=[f"TfidVect_{w}" for w in tfidf_vec.get_feature_names_out()]
        )
        ana = pd.concat([ana, count_df], axis=1)

In [21]:
data = df.merge(ana, how='right', on='participant')

In [22]:
data.columns

Index(['participant', 'timestamp', 'clinical-timestamp', 'sis-01', 'sis-02',
       'sis-03', 'sis-04', 'sis-05', 'sis-06', 'sis',
       ...
       'TfidVect_secondary', 'TfidVect_undergraduate', 'TfidVect_employed',
       'TfidVect_part', 'TfidVect_retired', 'TfidVect_time', 'TfidVect_asian',
       'TfidVect_black', 'TfidVect_caucasian', 'TfidVect_south'],
      dtype='object', length=143)

In [23]:
# Compute quartiles for discretization
siss_q1, siss_q2, siss_q3 = np.percentile(data["sis"], [25, 50, 75])
ohss_q1, ohss_q2, ohss_q3 = np.percentile(data["ohs"], [25, 50, 75])
okss_q1, okss_q2, okss_q3 = np.percentile(data["oks"], [25, 50, 75])

In [24]:
# Define quartile-based bins and labels
quartile_labels = [0, 1, 2, 3]

In [25]:
# Apply discretization
data["SISS_Category_Q"] = pd.cut(data["sis"], bins=[data["sis"].min(), siss_q1, siss_q2, siss_q3, data["sis"].max()],
                               labels=quartile_labels, include_lowest=True).astype(int)
data["OHSS_Category_Q"] = pd.cut(data["ohs"], bins=[data["ohs"].min(), ohss_q1, ohss_q2, ohss_q3, data["ohs"].max()],
                               labels=quartile_labels, include_lowest=True).astype(int)

data["OKSS_Category_Q"] = pd.cut(data["oks"], bins=[data["oks"].min(), okss_q1, okss_q2, okss_q3, data["oks"].max()],
                               labels=quartile_labels, include_lowest=True).astype(int)

In [26]:
# Extract features, target, and patient IDs for LOPO
X = data.iloc[:, -46:]
groups = data["participant"]

In [27]:
# Define classifier and hyperparameter grid

TABULAR_MODELS = [
    {
        "name": "XGBoost",
        "mode": ["classification", "regression"],
        "data_kind": "tabular",
        "estimator": XGBClassifier(
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=seed
        ),
        "param_grid": {
            "n_estimators": [20, 50, 100],
            "max_depth": [3, 6, 8],
            "learning_rate": [0.001, 0.01, 0.1, 0.3]
            #"subsample": [0.8, 1.0],
            #"colsample_bytree": [0.8, 1.0],
            #"reg_lambda": [1.0, 5.0],
        },
    },
    
    {
        "name": "CatBoost",
        "mode": ["classification", "regression"],
        "data_kind": "tabular",
        "estimator": CatBoostClassifier(
            verbose=False,
            random_seed=seed
        ),
        "param_grid": {
            "iterations": [50, 100],
            "depth": [3, 6, 8],
            "learning_rate": [0.001, 0.01, 0.1, 0.3]
            #"l2_leaf_reg": [3, 10],
        },
    },
    
    {
        "name": "LightGBM",
        "mode": ["classification", "regression"],
        "data_kind": "tabular",
        "estimator": LGBMClassifier(
            random_state=seed
        ),
        "param_grid": {
            "n_estimators": [20, 50, 100],
            #"num_leaves": [15, 31, 63],
            "max_depth": [3, 6, 8],
            "learning_rate": [0.001, 0.01, 0.1, 0.3]
            #"subsample": [0.8, 1.0],
            #"colsample_bytree": [0.8, 1.0],
        },
    },
    
    {
        "name": "DecisionTree (DT)",
        "mode": ["classification", "regression"],
        "data_kind": "tabular",
        "estimator": DecisionTreeClassifier(random_state=42),
        "param_grid": {
            "max_depth": [3, 6, 8]
            #"min_samples_split": [2, 5, 10],
            #"min_samples_leaf": [1, 2, 5],
            #"criterion": ["gini", "entropy"],
        },
    },

    # SVR è un regressore, Support Vector Regressor (attenzione: non va con scoring f1_macro!)
    {
        "name": "SVR",
        "mode": ["regression"],
        "data_kind": "tabular",
        "estimator": SVR(),
        "param_grid": {
            "kernel": ["rbf", "linear"],  ## "poly" --> sconsigliato per fare SVR
            "C": [0.1, 1, 10],
            "epsilon": [0.01, 0.1, 0.2],
            "gamma": ["scale", "auto"],
        },
    },

    # SVC, classificatore basato su SVM

    {
        "name": "SVC",
        "mode": ["classification"],
        "data_kind": "tabular",
        "estimator": SVC(probability=True),
        "param_grid": {
            "kernel": ["rbf", "linear", "poly"],
            "C": [0.1, 1, 10, 100],
            "gamma": ["scale", "auto"],
            "degree": [2, 3],
        },
    }
]

In [28]:
ANN_MODELS = [
    {
        "name": "LSTM",
        "mode": ["classification", "regression"],
        "estimator": KerasClassifier(
            model=build_lstm_model,
            epochs=10,
            batch_size=32,
            verbose=0
        ),
        "param_grid": {
            "model__n_units": [16, 32, 64],
            "model__dropout": [0.0, 0.2],
            "model__lr": [1e-3, 3e-4],
            "batch_size": [16, 32],
            "epochs": [10, 20],
        },
    },
    {
        "name": "TCN"                                 ##Temporal Convolutional Network
        "mode": ["classification", "regression"],
        "estimator": KerasClassifier(
            model=build_tcn_model,
            epochs=10,
            batch_size=32,
            verbose=0
        ),
        "param_grid": {
            "model__n_filters": [16, 32, 64],
            "model__kernel_size": [3, 5],
            "model__lr": [1e-3, 3e-4],
            "batch_size": [16, 32],
            "epochs": [10, 20],
        },
    },
]

In [29]:
OTHER_MODELS = [
    {
        "name": "ROCKET + RidgeClassifier",
        "mode": ["classification", "regression"],
        "estimator": Pipeline(steps=[
            ("rocket", Rocket(random_state=seed)),
            ("clf", RidgeClassifier())
        ]),
        "param_grid": {
            "rocket__num_kernels": [2000, 10000],
            "clf__alpha": [0.1, 1.0, 10.0],
        },
    }
]

In [30]:
MODELS = TABULAR_MODELS + ANN_MODELS + OTHER_MODELS

In [31]:
#MODELS

In [32]:
class_responses = ["OHSS_Category_Q", "SISS_Category_Q", "OKSS_Category_Q"]
reg_responses = ["ohs", "sis", "oks"]

target = {"classification": class_responses,
          "regression": reg_responses}

In [33]:
# Leave-One-Patient-Out CV (LOPO): OUTER LOPO, perchè ha più senso fare la predict su un solo paziente nel testing_set
outer_logo = LeaveOneGroupOut()

In [34]:
# Initialize results storage
overall_conf_matrix = np.zeros((4, 4))  # Assuming 4 categories (0, 1, 2, 3)

In [35]:
inner_ParOpt = {"LOPO": inner_LoPo_ParOpt, 
                "K-Fold": inner_GridSearch_ParOpt}

In [36]:
modes = ["classification", "regression"]

In [56]:
# Outer LOPO Loop
count=0

for mod in modes:

    responses = target[mod]

    for resp in responses:

        y = data[resp]

        performance_metrics_cla = []
        performance_metrics_reg = []
                
        for model_dict in MODELS:

            model = model_dict['estimator']
            param_grid = model_dict['param_grid']
            model_mod = model_dict['mode']
            print(model_dict['name'])

            if not mod in model_mod:

                continue

            # Initialize results storage
            #overall_conf_matrix = np.zeros((4, 4))  # Assuming 4 categories (0, 1, 2, 3)
            performance_metrics = []

            for train_idx, test_idx in outer_logo.split(X, y, groups):
                #print(train_idx) index
                count=count+1
                print(count)

                if count > 2:   ### TO-DO: per il debugging, fa solo la outer_lopo su uno dei pazienti

                    continue
                
                X_train_outer, X_test = X.iloc[train_idx].to_numpy(), X.iloc[test_idx].to_numpy()
                y_train_outer, y_test = y.iloc[train_idx].to_numpy(), y.iloc[test_idx].to_numpy()
                groups_train_outer = groups.iloc[train_idx]
                print(groups_train_outer.unique())

                for param_opt in inner_ParOpt.keys():

                    func = inner_ParOpt[param_opt]
                    print(param_opt)

                    ## INNER OPTIMIZATION (best FOR each group-iteration) 
                    best_model, best_params, best_score = func(
                                                                model,
                                                                param_grid,
                                                                X_train_outer,
                                                                y_train_outer,
                                                                groups_train_outer
                                                            )
                    
                    # Train best model on full outer training set
                    #if model_dict['name'] in [...]:
                    #    model.fit(X,
                    #              y,
                    #              epochs=10,
                    #              batch_size=32,
                    #              validation_split=0.2)
                    #else:
                    best_model.fit(X_train_outer, y_train_outer)
                    
                    y_pred = best_model.predict(X_test)
        
                    if mod == "classification":

                        print(mod)
                        # Compute metrics CLASSIFICATION
                        try:
                            f1 = f1_score(y_test, y_pred, average="macro")
                        except:
                            f1 = np.nan
                        balanced_acc = balanced_accuracy_score(y_test, y_pred)
                        recall = recall_score(y_test, y_pred, average="macro")
                        precision = precision_score(y_test, y_pred, average="macro")
                        conf_matrix = confusion_matrix(y_test, y_pred, labels=[0, 1, 2, 3])
                    
                        # Aggregate confusion matrices
                        #overall_conf_matrix += conf_matrix
                    
                        # Store results
                        performance_metrics_cla.append([model_dict['name'], param_opt, f1, balanced_acc, recall, precision])
    
                    if mod == "regression":

                        print(mod)
                        # Compute metrics REGRESSION
                        mae = mean_absolute_error(y_test, y_pred)
                        mse = mean_squared_error(y_test, y_pred)
                        rmse = np.sqrt(mse)
                        r2 = r2_score(y_test, y_pred)
                        mape = mean_absolute_percentage_error(y_test, y_pred)
                        medae = median_absolute_error(y_test, y_pred)
                        evs = explained_variance_score(y_test, y_pred)
    
                        # Store results
                        performance_metrics_reg.append([model_dict['name'], param_opt, mae, mse, rmse, r2, mape, medae, evs])

        print('entri in IF')
        
        if mod == "classification":
            print('group_by')
            performance_df_raw = pd.DataFrame(performance_metrics_cla, columns=["Model", "CV", "Macro-F1", "Balanced Accuracy", "Macro Recall", "Macro Precision"])
            performance_df = (performance_df_raw.groupby(["Model", "CV"])[["Macro-F1", "Balanced Accuracy", "Macro Recall", "Macro Precision"]]
                                                               .mean()
                                                               .reset_index()
                                                               #.rename(columns={"score": "mean_score"})
                                                               )

        else:
            print('group_by')
            performance_df_raw = pd.DataFrame(performance_metrics_reg, columns=["Model", "CV", "MAE", "MSE", "RMSE", "R2", "MAPE", "MEDAE", "EVS"])
            performance_df = (performance_df_raw.groupby(["Model", "CV"])[["MAE", "MSE", "RMSE", "R2", "MAPE", "MEDAE", "EVS"]]
                                                               .mean()
                                                               .reset_index()
                                                               #.rename(columns={"score": "mean_score"})
                                                               )

        # Save performance metrics and overall confusion matrix
        output_path = os.path.join(root, "new_results/results_" + mod + ".xlsx")
        with pd.ExcelWriter(output_path) as writer:
            performance_df.to_excel(writer, sheet_name=resp)

        #rc = save_ConfMat(overall_conf_matrix, root)

XGBoost
1
[ 2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]
LOPO


KeyboardInterrupt: 

In [47]:
performance_metrics

[]